# Predicting NBA All-Star Selection

Logistic regression **implemented from scratch** (numpy only — no sklearn), applied
to a real prediction task. All design decisions are recorded with their reasoning in
[`docs/DATA_DECISIONS.md`](../docs/DATA_DECISIONS.md).

**Task.** Binary classification: given one player's statistics for one NBA regular
season (2000–2025), predict whether he was named an All-Star that season.
One row = one player-season.

**Input.** `x ∈ ℝ⁶`, standardised (zero mean, unit variance, fitted on the training
split only): `mp` (availability), `per` (production quality), `usg_percent` (role),
`ws_48` (rate value), `age` (career arc), `win_rate` (team success, minutes-weighted
across stints for traded players).

**Output.** `ŷ = σ(wᵀx + b) ∈ (0, 1)` — the estimated probability of selection.
A threshold τ, chosen on the validation split, converts it to a yes/no prediction.
τ = 0.5 is *not* assumed: only ~7% of eligible player-seasons are All-Stars.

**Data.** Kaggle "NBA Stats (1947–present)": four CSVs. `Advanced.csv` (features),
`All-Star Selections.csv` (labels — every named All-Star is y = 1, including injured
selectees), `Team Summaries.csv` (team win rate), `Per 100 Poss.csv` (loaded, unused).

## Setup — self-contained bootstrap

The cell below makes the notebook runnable anywhere with nothing pre-arranged:
on a fresh Colab kernel it clones the public repo (the real code lives in
importable modules under `src/` — this notebook only orchestrates them) and
downloads the pinned data snapshot from the repo's `data-v1` GitHub release;
run locally inside the repo, both steps detect what is already present and do
nothing. The only dependencies are numpy, pandas and matplotlib, which Colab
preinstalls — no `pip install` is needed there.

In [ ]:
import os, pathlib, subprocess, sys, urllib.request, zipfile

REPO_URL = "https://github.com/grishma34/nba-allstar"
DATA_URL = REPO_URL + "/releases/download/data-v1/nba-data.zip"
CSVS = ["Advanced.csv", "All-Star Selections.csv",
        "Per 100 Poss.csv", "Team Summaries.csv"]

# 1. Code. If src/ is nowhere in sight this is a fresh kernel (e.g. Colab):
#    clone the repo and work inside it.
if not pathlib.Path("src").exists() and not pathlib.Path("../src").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    os.chdir("nba-allstar")

# Works from the repo root or from notebooks/: find the root, put it on sys.path.
ROOT = pathlib.Path.cwd() if pathlib.Path("src").exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))

# 2. Data. The CSVs are git-ignored, so a fresh clone has none: fetch the
#    pinned release snapshot. Present already -> do nothing.
DATA_DIR = ROOT / "data"
if not all((DATA_DIR / name).exists() for name in CSVS):
    print("downloading data snapshot:", DATA_URL)
    zip_path, _ = urllib.request.urlretrieve(DATA_URL)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_DIR)
DATA_DIR = str(DATA_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_raw, build_dataset, load_team_summaries, add_team_win_rate
from src.features import select_features, chronological_split, standardise
from src.model import predict_proba, log_loss
from src.train import fit
from src import evaluate as ev
from src import plots

print("numpy", np.__version__, "| pandas", pd.__version__)

## Build the dataset

`build_dataset()` applies the recorded decisions: NBA only, seasons 2000–2025
(no 1999 All-Star Game; 2026 incomplete), traded players resolved to their
season-combined `2TM`-style row, labels joined on `(player_id, season)`, and the
eligibility threshold **mp ≥ 500** — which removes every row with a missing feature
and drops exactly four All-Stars (printed below by name, never silently: all four are
`replaced=True` injury selections with under 310 minutes). `add_team_win_rate()`
attaches the minutes-weighted team win rate.

In [ ]:
advanced, allstar, per100 = load_raw(DATA_DIR)
df = build_dataset(advanced, allstar)
df = add_team_win_rate(df, load_team_summaries(DATA_DIR), advanced)
df[["player", "season", "team", "mp", "per", "usg_percent", "ws_48", "age", "win_rate", "y"]].head()

## Features, split, standardisation

**Chronological split** (train 2000–17, validation 2018–21, test 2022–25): All-Star
selection is comparative — ~24 slots per season — so a random split would leak
season-level competitiveness across the boundary; chronological splitting also
mirrors deployment. **Standardisation is fitted on train only** and applied to
validation and test; fitting on all data would leak test information into training.

In [ ]:
FEATURES = ["mp", "per", "usg_percent", "ws_48", "age", "win_rate"]

train_df, val_df, test_df = chronological_split(df, train_end=2017, val_end=2021)
X_tr, y_tr, meta_tr = select_features(train_df, FEATURES)
X_va, y_va, meta_va = select_features(val_df, FEATURES)
X_te, y_te, meta_te = select_features(test_df, FEATURES)
X_tr, X_va, X_te, scaler = standardise(X_tr, X_va, X_te)

for name, X, y in [("train", X_tr, y_tr), ("val", X_va, y_va), ("test", X_te, y_te)]:
    print(f"{name:5s}: X {X.shape}, positives {int(y.sum()):3d} ({100 * y.mean():.1f}%)")

## Train

Batch gradient descent on the binary cross-entropy loss
`L = −(1/n) Σ [y·log ŷ + (1−y)·log(1−ŷ)]`, with the gradients
`∂L/∂w = (1/n)Xᵀ(ŷ−y)` and `∂L/∂b = (1/n)Σ(ŷ−y)` derived by hand in
`src/model.py` and verified against numerical finite differences.
Learning rate 1.0 (swept 0.01–10: everything in 0.1–3 reaches the same optimum —
the loss is convex — and 10 oscillates); 5,000 iterations (converged by ~600;
the marked validation minimum is where early stopping would halt).

In [ ]:
w, b, history = fit(X_tr, y_tr, X_va, y_va, lr=1.0, n_iters=5000, verbose=False)
print("weights:", {f: round(wi, 3) for f, wi in zip(FEATURES, w)}, "| bias:", round(b, 3))
plots.loss_curves(history)

## Evaluate

Accuracy is not a primary metric: predicting "no" for everyone is ~93% accurate and
useless. We report log loss (the objective), precision/recall/F1 at a justified
threshold, PR-AUC (the informative area under imbalance — random scores the base
rate, not 0.5), ROC-AUC, and calibration. **τ is chosen on the validation split**
by maximising F1; the test set plays no part in the choice.

In [ ]:
p_va = predict_proba(X_va, w, b)
taus = np.arange(0.10, 0.75, 0.05)
sweep = pd.DataFrame(
    [{"tau": round(t, 2),
      **dict(zip(["precision", "recall", "f1"],
                 np.round(ev.precision_recall_f1(y_va, ev.binarise(p_va, t)), 3)))}
     for t in taus])
TAU = float(sweep.loc[sweep["f1"].idxmax(), "tau"])
print(sweep.to_string(index=False))
print(f"\nchosen tau = {TAU} (max F1 on validation)")

In [ ]:
p_tr = predict_proba(X_tr, w, b)
p_te = predict_proba(X_te, w, b)

# Baselines: the log-loss-optimal constant, and the same hand-written model
# trained on a single feature (vorp) — no extra machinery needed for either.
base_te = ev.baseline_constant_rate(y_tr, len(y_te))
Xv_tr, _, _ = select_features(train_df, ["vorp"])
Xv_va, _, _ = select_features(val_df, ["vorp"])
Xv_te, _, _ = select_features(test_df, ["vorp"])
Xv_tr, Xv_va, Xv_te, _ = standardise(Xv_tr, Xv_va, Xv_te)
wv, bv, _ = fit(Xv_tr, y_tr, Xv_va, y_va, lr=1.0, n_iters=5000, verbose=False)
pv_te = predict_proba(Xv_te, wv, bv)

print(ev.loss_table({
    "train (model)": (y_tr, p_tr),
    "val (model)": (y_va, p_va),
    "test (model)": (y_te, p_te),
    "test (constant-rate baseline)": (y_te, base_te),
    "test (vorp-alone baseline)": (y_te, pv_te),
}).to_string(index=False))

yhat_te = ev.binarise(p_te, TAU)
print("\nconfusion (test):", ev.confusion(y_te, yhat_te))
p, r, f1 = ev.precision_recall_f1(y_te, yhat_te)
print(f"precision {p:.3f}  recall {r:.3f}  F1 {f1:.3f}   (majority-class baseline: 0/0/0)")
print(f"ROC-AUC {ev.roc_auc(y_te, p_te):.4f}   PR-AUC {ev.pr_auc(y_te, p_te):.4f}"
      f"   (random PR-AUC = base rate = {y_te.mean():.3f})")

In [ ]:
prec, rec, _ = ev.pr_curve(y_te, p_te)
plots.pr_plot(prec, rec, auc=ev.pr_auc(y_te, p_te), base_rate=float(y_te.mean()))
fpr, tpr, _ = ev.roc_curve(y_te, p_te)
plots.roc_plot(fpr, tpr, auc=ev.roc_auc(y_te, p_te))

In [ ]:
calib = ev.calibration_table(y_te, p_te)
plots.calibration_plot(calib)
calib.round(3)

## Named errors — where the loss and the task disagree

The most confident mistakes, with names. The investigation (full write-up in
`docs/DATA_DECISIONS.md`) tested the obvious hypothesis — that team success is the
missing variable — and **refuted it**: win rate barely separates the error groups
(means .552 vs .556). What separates them is **career trajectory**: the false
positives are established stars around age 30 passed over by voters, the false
negatives are ascending young stars (mean age 24) selected ahead of their numbers,
plus defence-first picks whose value sits in columns the feature set deliberately
excludes. A single linear age term cannot represent both directions at once — the
effect at the boundary is an interaction, outside the hypothesis space. The scatter
below shows the refutation visually: All-Stars and high-probability non-selections
mix across the entire win-rate axis.

In [ ]:
wr_te = test_df.reset_index(drop=True)["win_rate"].to_numpy()
plots.winrate_scatter(wr_te, p_te, y_te, threshold=TAU)
ev.named_errors(meta_te, y_te, p_te, threshold=TAU, top_n=10).round(3)

## Findings and limitations

**Findings.** Test log loss 0.077 vs 0.258 for the best constant (features matter)
and 0.107 for vorp alone (breadth matters). Precision 0.75 / recall 0.77 at
τ = 0.35; PR-AUC 0.86 against a 0.07 random baseline. Adding team win rate was
adopted on experiment (log loss −9%, +10 All-Stars found); the production × win-rate
interaction was tested and **rejected** — the team-success effect is additive.
The named errors follow a trajectory pattern the linear model cannot represent.

**Limitations.** (1) *Temporal leakage*: selection happens mid-season, features are
full-season — the model is retrospective analysis, not live prediction.
(2) *Linear hypothesis space*: interactions like age × production are unrepresentable
unless built as features. (3) *Label ambiguity*: "named All-Star" mixes voted-in
players, injured selectees and commissioner replacements, which the data cannot
separate. (4) *No defensive signal*: `dbpm`/`dws` were excluded for interpretability;
the missed defence-first selections are the measured cost. (5) Four All-Stars with
< 500 minutes are outside the eligible population by construction.